# **Pull GitHub Repository**

In [ ]:
!pip install -q torchmetrics timm

In [ ]:
from google.colab import userdata

!rm -rf /content/ECM3401_Individual_Project

token = userdata.get('GitHub')
!git clone -b causal https://{token}@github.com/sccthomas/ECM3401_Individual_Project.git

In [ ]:
import sys

sys.path.append('/content/ECM3401_Individual_Project/')
!ls /content/ECM3401_Individual_Project/src/

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# **Define the Model**

In [1]:
import torch
from src.vision_transformer.model import SemanticSegmentationVisionTransformer
from src.self_supervised_learning.contrastive_loss import ContrastivePreTraining
from src.self_supervised_learning.masked_region_loss import MaskedRegionLoss

# --------------------------------------------
# Parameters
# --------------------------------------------
device = torch.device("cuda")
metric_device = torch.device("cpu")

image_dims = (3, 256, 256)  # Input image dimensions
patch_embedding_scale_1 = (16, 768)  # Patch size and embedding dimension for scale 1
patch_embedding_scale_2 = (8, 512)  # Patch size and embedding dimension for scale 2
patch_embedding_scale_3 = (4, 256)  # Patch size and embedding dimension for scale 3

# --------------------------------------------
# Model Initialization
# --------------------------------------------
model = SemanticSegmentationVisionTransformer(
    # - Image dimensions
    image_dims=image_dims,
    # - Hyper Parameters
    num_encoder_layers=6,
    use_heavyweight_decoder=False,
    use_swin_transformer=False,
    use_skip_layer_gated_attention=False,
    use_learnable_skip_layers=True,
    skip_layer_ratio=2,
    encoder_dropout_rate=0.2,
    patch_fusion_dropout_rate=0.2,
    decoder_dropout_rate=0.2,
    num_encoder_heads=8,
    num_classes=1,
    # - Scales
    patch_embedding_scale_1=patch_embedding_scale_1,
    patch_embedding_scale_2=patch_embedding_scale_2,
    patch_embedding_scale_3=patch_embedding_scale_3,
).to(device)

ssl_model_mea = MaskedRegionLoss(
    model=model,
    max_patch_size=16,
).to(device)

ssl_model_con = ContrastivePreTraining(
    model=model,
    encoder_dims=[768, 512, 256],
    projection_dim=128,
).to(device)

ssl_model = ssl_model_con

In [ ]:
ssl_model

In [ ]:
model

## **Optional Load Model**

In [ ]:
import torch

# Load the model's state_dict
# path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/generative_ssl/version_1/model_pre_fine_tune.pth"
path = "/content/drive/MyDrive/best_model_ssl.pth"
checkpoint = torch.load(path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint)

# Load the SSL model's state_dict
# path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/generative_ssl/version_1/ssl_model.pth"
path = "/content/drive/MyDrive/best_ssl_contrastive_model.pth"
checkpoint = torch.load(path, map_location=device, weights_only=True)
ssl_model.load_state_dict(checkpoint)

# **Train the Model With SSL**

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from src.dataset.snow import SnowDataset
from src.training.self_supervised_learning import train_model

# --------------------------------------------
# Parameters
# --------------------------------------------
dataset_dir = "/content/drive/MyDrive/snow_dataset"  # Replace with your dataset path
batch_size = 32
num_epochs = 50
learning_rate = 1e-4
patience = 5  # Early stopping patience

# --------------------------------------------
# Dataset and DataLoader
# --------------------------------------------

# Load the dataset and split it into train and validation sets
train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    normalize=True,
    resize=True,
    rotate=True,
)
print("Length of dataset:", len(train_dataset))
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
# --------------------------------------------
# Loss, Optimizer, and Scheduler
# --------------------------------------------
optimizer = optim.AdamW(ssl_model.parameters(), lr=learning_rate, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs // 2, T_mult=1)

# --------------------------------------------
# Mixed Precision Setup
# --------------------------------------------
scaler = torch.amp.GradScaler(device.type)

# --------------------------------------------
# Train Model
# --------------------------------------------
train_model(
    ssl_model=ssl_model,
    num_epochs=num_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_loader=train_loader,
    val_loader=val_loader,
    patience=patience,
    device=device,
)

In [ ]:
!cp /content/best_model_ssl.pth /content/drive/MyDrive/best_ssl_mea_model.pth

In [ ]:
!cp /content/best_model.pth /content/drive/MyDrive/best_model_ssl_mea.pth

# **Evaluate the Model**

In [2]:
import torch

# Load the model's state_dict
path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/contrastive_ssl/classic/model_pre_fine_tune.pth"
# path = "/content/drive/MyDrive/best_model_ssl.pth"
checkpoint = torch.load(path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint)
model = model.eval()

# Load the SSL model's state_dict
path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/contrastive_ssl/classic/contrastive_ssl_model.pth"
# path = "/content/drive/MyDrive/best_ssl_contrastive_model.pth"
checkpoint = torch.load(path, map_location=device, weights_only=True)
ssl_model.load_state_dict(checkpoint)
ssl_model = ssl_model.eval()

In [3]:
from src.dataset.snow import SnowDataset
from torch.utils.data import DataLoader

dataset_dir = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/snow_dataset"
# dataset_dir = "/content/drive/MyDrive/snow_dataset"

# Dataset and DataLoader
batch_size = 5

train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=30,
    resize=True,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
)
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)

In [4]:
from src.training.evaluation import (
    evaluate_with_no_modifications, evaluate_with_illumination_modifications, evaluate_with_background_modifications,
    evaluate_with_texture_modifications
)

In [5]:
import matplotlib.pyplot as plt

path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/contrastive_ssl/classic/figures/pre_fine_tune/no_image_modifications"
for idx in range(batch_size):
    evaluate_with_no_modifications(
        model,
        images[idx],
        masks[idx],
        device,
        path=path,
        name=f"image_{idx + 1}"
    )
    print("\n" * 5)
    plt.close('all')

In [6]:
import matplotlib.pyplot as plt

path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/contrastive_ssl/classic/figures/pre_fine_tune/illumination_modifications"
for idx in range(batch_size):
    evaluate_with_illumination_modifications(
        model,
        images[idx],
        masks[idx],
        device,
        path=path,
        name=f"image_{idx + 1}"
    )
    print("\n" * 5)
    plt.close('all')

In [7]:
import matplotlib.pyplot as plt

path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/contrastive_ssl/classic/figures/pre_fine_tune/background_modifications"

for dir_, mtype in [("contrast", "Contrast"), ("gaussian_blur", "Gaussian Blur"), ("gaussian_noise", "Gaussian Noise"),
                    ("invert", "Invert"), ("sharpness", "Sharpness"), ("simple", "Simple")]:
    for idx in range(batch_size):
        evaluate_with_background_modifications(
            model,
            images[idx],
            masks[idx],
            device,
            mtype=mtype,
            path=f'{path}/{dir_}',
            name=f"image_{idx + 1}"
        )
        plt.close('all')


In [ ]:
import matplotlib.pyplot as plt

path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/contrastive_ssl/classic/figures/pre_fine_tune/texture_modifications"
for dir_, texture_type in [("staining", "Staining"), ("background_artifacts", "Background Artifacts")]:
    for idx in range(batch_size):
        evaluate_with_texture_modifications(
            model,
            images[idx],
            masks[idx],
            device,
            texture_type=texture_type,
            path=f'{path}/{dir_}',
            name=f"image_{idx + 1}"
        )
        print("\n" * 5)
        plt.close('all')

## **Contrastive SSL**

In [ ]:
from src.self_supervised_learning.contrastive_loss import visualize_tsne

visualize_tsne(ssl_model, images)

## **Masked Region SSL**

In [ ]:
from src.self_supervised_learning.masked_region_loss import visualise_masked_region_prediction

for idx in range(batch_size):
    visualise_masked_region_prediction(ssl_model, images[idx])